# Sensitivity Analysis (SA) ABM

This notebook runs sensitivity analysis over two model parameters:
- **deterioration_step**: how much compliance worsens when ML suggests deterioration 
- **improvement_step**: how much compliance improves when ML suggests improvement 

Scenarios:
- **random**: random inspections
- **equal**: equal-weight risk score
- **tuned**: tuned risk score weights

Outputs:
- caught rate summary (mean ± 95% CI)
- compliance distribution plots
- combined 3-panel plots 
- Excel export of summaries + time series

Seed policy:
- **POP_SEED** fixes the sampled population (same FBO sample across runs)
- **run_seed** varies stochasticity inside the simulation (replications)


In [ ]:
!pip install -r requirements.txt >nul 2>&1

In [ ]:
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from math import sqrt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector
from ml_inference import predict_inspection_result
from IPython.display import display, Markdown

In [ ]:
# Config

EXCEL_FILE = "init_ABM_dataset.xlsx"

GRID_W, GRID_H = 90, 55
TICK_DAYS = 90
SIM_START_DATE = datetime.today().date()

# Experiment design
N_FBO = 1000
N_TICKS = 20
N_REPS = 10
SCENARIOS = ("random", "equal", "tuned")

DELTA_GRID = (0.05, 0.10, 0.15)   # deterioration_step
IMPROVE_GRID = (1, 2)            # improvement_step

# Seed policy
POP_SEED = 42 

# Scenario 
SCENARIO_CONFIG = {
    "random": {
        "weights": dict(size=0, former=0, type=0, product=0, interval=0),
        "random_flag": True,
    },
    "equal": {
        "weights": dict(size=1, former=1, type=1, product=1, interval=1),
        "random_flag": False,
    },
    "tuned": {
        "weights": dict(size=0, former=1.5, type=-1, product=1, interval=2),
        "random_flag": False,
    },
}
# Outputs
OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figs")
EXCEL_OUT = os.path.join(OUT_DIR, "compliance_sensitivity_timeseries.xlsx")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Load data

_DF_CACHE = None

def load_rows_with_date(path, n=500, sim_start_date=None, sample_seed=POP_SEED):
    global _DF_CACHE

    # Read Excel only once
    if _DF_CACHE is None:
        _DF_CACHE = pd.read_excel(path, parse_dates=["Last Inspection date"])

    df = _DF_CACHE

    required = {
        "name1","cvr_number","FBO type","Product type","Operation type",
        "FBO Size","Inspection result","Last Inspection date"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    if sim_start_date is None:
        sim_start_date = pd.Timestamp("today").normalize().date()

    # sample fixed population (if needed)
    if n < len(df):
        df_sample = df.sample(n, random_state=sample_seed)
    else:
        df_sample = df

    items = []
    for idx, r in df_sample.iterrows():
        last_date = pd.to_datetime(r["Last Inspection date"]).date()
        interval_days = max(0, (sim_start_date - last_date).days)
        items.append({
            "agent_id": f"{r['name1']}|{r['cvr_number']}|{idx}",
            "FBO type": int(r["FBO type"]),
            "Product type": int(r["Product type"]),
            "Operation type": int(r["Operation type"]),
            "FBO Size": int(r["FBO Size"]),
            "inspection_result": int(r["Inspection result"]),
            "last_inspection_date": last_date,
            "inspection_interval_days": int(interval_days),
        })
    return items



In [ ]:
# ABM model

class FBOAgent(Agent):
    def __init__(self, uid, model, a):
        super().__init__(uid, model)
        self.agent_id = a["agent_id"]
        self.FBO_type = int(a["FBO type"])
        self.Product_type = int(a["Product type"])
        self.Operation_type = int(a["Operation type"])
        self.FBO_size = int(a["FBO Size"])
        self.inspection_result = int(a["inspection_result"])

        self.compliance_level = self.inspection_result
        self.actual_compliance_level = self.compliance_level
        self.former_inspection_result = self.inspection_result
        self.last_inspection_date = a["last_inspection_date"]
        self.inspection_interval_days = int(a["inspection_interval_days"])
        self.color = {1:"green", 2:"yellow", 3:"orange", 4:"red"}[self.compliance_level]

    def pre_ml_tick_update(self):
        features = {
            "FBO type": self.FBO_type,
            "Product type": self.Product_type,
            "Operation type": self.Operation_type,
            "Former Inspection result": self.former_inspection_result,
            "FBO Size": self.FBO_size,
            "Inspection interval (days)": self.inspection_interval_days,
        }
        ch = predict_inspection_result(features)

        if ch == 1 and self.inspection_interval_days != 90:
            step = self.model.deterioration_step
            self.actual_compliance_level = min(4, self.actual_compliance_level + step) # Deterioration step
            self.compliance_level = round(self.actual_compliance_level)

        self.color = {1:"green", 2:"yellow", 3:"orange", 4:"red"}[self.compliance_level]

    def inspect_and_maybe_improve(self):
        self.last_inspection_date = self.model.current_date
        self.inspection_result = self.compliance_level

        if self.compliance_level > 1:
            self.model.at_risk_inspected_this_tick += 1

        features = {
            "FBO type": self.FBO_type,
            "Product type": self.Product_type,
            "Operation type": self.Operation_type,
            "Former Inspection result": self.former_inspection_result,
            "FBO Size": self.FBO_size,
            "Inspection interval (days)": self.inspection_interval_days,
        }
        ch = predict_inspection_result(features)

        if ch == -1:
            step = self.model.improvement_step
            self.compliance_level = max(1, self.compliance_level - step)  # Improvement step

        self.color = {1:"green", 2:"yellow", 3:"orange", 4:"red"}[self.compliance_level]
        self.actual_compliance_level = self.compliance_level
        self.inspection_interval_days = 90
        self.former_inspection_result = self.inspection_result


class InspectorAgent(Agent):
    def __init__(self, uid, model, move_range, sight, inspection_capacity):
        super().__init__(uid, model)
        self.move_range = move_range
        self.sight = sight
        self.inspection_capacity = inspection_capacity

    def step(self):
        # Move with several tries to avoid overlaps
        for _ in range(12):
            dx = random.randint(-self.move_range, self.move_range)
            dy = random.randint(-self.move_range, self.move_range)
            nx = (self.pos[0] + dx) % self.model.grid.width
            ny = (self.pos[1] + dy) % self.model.grid.height
            if self.model.grid.is_cell_empty((nx, ny)):
                break
        self.model.grid.move_agent(self, (nx, ny))

        nb = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False, radius=self.sight)
        fbos = [a for a in nb if isinstance(a, FBOAgent)]
        if not fbos:
            return

        def scale(arr):
            a = np.array(arr, dtype=float)
            mn, mx = a.min(), a.max()
            return np.full_like(a, 2.5) if mx == mn else 1 + 3*(a - mn)/(mx - mn)

        sizes     = scale([f.FBO_size for f in fbos])
        formers   = scale([f.former_inspection_result for f in fbos])
        types     = scale([f.FBO_type for f in fbos])
        products  = scale([f.Product_type for f in fbos])
        intervals = scale([f.inspection_interval_days for f in fbos])

        w = self.model.weights
        score = (
            w["size"] * sizes
            + w["former"] * formers
            + w["type"] * types
            + w["product"] * products
            + w["interval"] * intervals
        )

        random_condition = (
            self.model.random_inspection
            or all(self.model.weights[k] == 0 for k in ["size","former","type","product","interval"])
        )
        order = np.random.permutation(len(fbos)) if random_condition else np.argsort(-score)

        selected = []
        for idx in order:
            f = fbos[int(idx)]
            if f not in self.model.assigned_this_tick:
                selected.append(f)
                self.model.assigned_this_tick.add(f)
            if len(selected) >= self.inspection_capacity:
                break

        for f in selected:
            f.inspect_and_maybe_improve()
            self.model.inspected_ids_this_tick.append(f.agent_id)
            self.model.inspections_this_tick += 1


class InspectionModel(Model):
    def __init__(
        self, items, num_inspectors=8, width=GRID_W, height=GRID_H, start_date=SIM_START_DATE,
        w_size=1.0, w_former=1.0, w_type=1.0, w_product=1.0, w_interval=1.0,
        move_range=18, sight=20, inspection_capacity=25, random_inspection=False,
        deterioration_step=0.10, improvement_step=1
    ):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=True)
        self.schedule = RandomActivation(self)
        self.tick_days = TICK_DAYS
        self.current_date = start_date

        self.weights = {
            "size": w_size, "former": w_former, "type": w_type,
            "product": w_product, "interval": w_interval
        }
        self.random_inspection = random_inspection

        # Sensitivity parameters
        self.deterioration_step = deterioration_step
        self.improvement_step = improvement_step

        self.assigned_this_tick = set()
        self.inspected_ids_this_tick = []
        self.inspections_this_tick = 0
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

        # FBOs
        for a in items:
            uid = a["agent_id"]
            if uid in self.schedule._agents:
                continue
            f = FBOAgent(uid, self, a)
            x, y = self.random.randrange(width), self.random.randrange(height)
            self.grid.place_agent(f, (x, y))
            self.schedule.add(f)

        # Inspectors (place in empty cells)
        def _empty_cell():
            tries = 0
            while True:
                x, y = self.random.randrange(width), self.random.randrange(height)
                if self.grid.is_cell_empty((x, y)):
                    return x, y
                tries += 1
                if tries > 10000:
                    return x, y

        for i in range(num_inspectors):
            ins = InspectorAgent(f"INS_{i}", self, move_range, sight, inspection_capacity)
            x, y = _empty_cell()
            self.grid.place_agent(ins, (x, y))
            self.schedule.add(ins)

        # DataCollector
        self.datacollector = DataCollector(model_reporters={
            "compliance_level_1": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 1),
            "compliance_level_2": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 2),
            "compliance_level_3": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 3),
            "compliance_level_4": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 4),
            "inspections": lambda m: m.inspections_this_tick,

            "at_risk": lambda m: sum(1 for a in m.schedule.agents
                                     if isinstance(a, FBOAgent) and a.compliance_level > 1),

            "at_risk_start": lambda m: m.at_risk_this_tick,
            "at_risk_inspected": lambda m: m.at_risk_inspected_this_tick,

            "caught_rate": lambda m: (
                0.0 if m.at_risk_this_tick == 0
                else 100.0 * m.at_risk_inspected_this_tick / m.at_risk_this_tick
            ),
        })

    def step(self):
        self.current_date = self.current_date + timedelta(days=self.tick_days)

        # Reset tick trackers
        self.assigned_this_tick.clear()
        self.inspected_ids_this_tick = []
        self.inspections_this_tick = 0
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

        # Pre-ML update + at-risk at start
        for a in list(self.schedule.agents):
            if isinstance(a, FBOAgent):
                a.pre_ml_tick_update()

        self.at_risk_this_tick = sum(
            1 for a in self.schedule.agents
            if isinstance(a, FBOAgent) and a.compliance_level > 1
        )

        # Inspectors
        for a in list(self.schedule.agents):
            if isinstance(a, InspectorAgent):
                a.step()

        # End-of-tick interval update for non-inspected
        inspected_set = set(self.inspected_ids_this_tick)
        for a in list(self.schedule.agents):
            if isinstance(a, FBOAgent) and a.agent_id not in inspected_set:
                a.inspection_interval_days += TICK_DAYS

        self.datacollector.collect(self)


In [ ]:
# Single run 

def run_single(run_seed, scenario, n_fbo=N_FBO, delta=0.10, improve_step=1, sample_seed=POP_SEED):
    """
    run_seed    -> seeds Python & NumPy for stochastic simulation behavior
    sample_seed -> seeds the sampled population (fixed across SA runs by default)
    delta       -> deterioration_step
    improve_step-> improvement_step
    """
    random.seed(run_seed)
    np.random.seed(run_seed)

    items = load_rows_with_date(EXCEL_FILE, n=n_fbo, sim_start_date=SIM_START_DATE, sample_seed=sample_seed)

    if scenario not in SCENARIO_CONFIG:
        raise ValueError(f"Unknown scenario: {scenario}. Use {tuple(SCENARIO_CONFIG.keys())}")

    cfg = SCENARIO_CONFIG[scenario]
    w = cfg["weights"]
    random_flag = cfg["random_flag"]

    m = InspectionModel(
        items,
        num_inspectors=8,
        w_size=w["size"], w_former=w["former"], w_type=w["type"],
        w_product=w["product"], w_interval=w["interval"],
        move_range=18, sight=20, inspection_capacity=25,
        random_inspection=random_flag,
        deterioration_step=delta,
        improvement_step=improve_step,
    )

    for _ in range(N_TICKS):
        m.step()

    df = (
        m.datacollector
         .get_model_vars_dataframe()
         .reset_index(drop=False)
         .rename(columns={"index": "tick"})
    )

    # Mean caught rate excluding ticks with 0 at-risk at start
    df_nonzero = df[df["at_risk_start"] > 0]
    caught_rate_mean = 0.0 if len(df_nonzero) == 0 else float(df_nonzero["caught_rate"].mean())

    ts = df[[
        "tick",
        "compliance_level_1", "compliance_level_2",
        "compliance_level_3", "compliance_level_4"
    ]].copy()

    return caught_rate_mean, ts


In [ ]:
# CI + SA run

def mean_ci(vals):
    a = np.asarray(vals, float)
    m = a.mean()
    if len(a) < 2:
        return m, 0.0
    s = a.std(ddof=1)
    half = 1.96 * s / sqrt(len(a))
    return m, half


def run_sensitivity_delta(delta_grid=DELTA_GRID, scenarios=SCENARIOS, n_reps=N_REPS, n_fbo=N_FBO):
    rows = []
    ts_store = {(d, sc): [] for d in delta_grid for sc in scenarios}

    for d in delta_grid:
        for sc in scenarios:
            caught_rates = []
            for r in range(n_reps):
                run_seed = 1000 + 37*r + int(d*100)  # reproducible but varied
                caught_rate_mean, ts = run_single(
                    run_seed, sc, n_fbo=n_fbo, delta=d, improve_step=1, sample_seed=POP_SEED
                )
                caught_rates.append(caught_rate_mean)
                ts_store[(d, sc)].append(ts)

            m, ci = mean_ci(caught_rates)
            rows.append({
                "param": "deterioration_step",
                "deterioration_step": d,
                "scenario": sc,
                "caught_rate_mean_%": round(m, 1),
                "ci_caught_rate_%": round(ci, 1),
                "n_reps": len(caught_rates),
            })

    return pd.DataFrame(rows), ts_store


def run_sensitivity_improve(improve_grid=IMPROVE_GRID, scenarios=SCENARIOS, n_reps=N_REPS, n_fbo=N_FBO):
    rows = []
    ts_store = {(step, sc): [] for step in improve_grid for sc in scenarios}

    for step in improve_grid:
        for sc in scenarios:
            caught_rates = []
            for r in range(n_reps):

                print(f"Running: improve_step={step}, scenario={sc}, rep={r+1}/{n_reps}")

                
                run_seed = 2000 + 41*r + step
                caught_rate_mean, ts = run_single(
                    run_seed, sc, n_fbo=n_fbo, delta=0.10, improve_step=step, sample_seed=POP_SEED
                )
                caught_rates.append(caught_rate_mean)
                ts_store[(step, sc)].append(ts)

            m, ci = mean_ci(caught_rates)
            rows.append({
                "param": "improvement_step",
                "improvement_step": step,
                "scenario": sc,
                "caught_rate_mean_%": round(m, 1),
                "ci_caught_rate_%": round(ci, 1),
                "n_reps": len(caught_rates),
            })

    return pd.DataFrame(rows), ts_store


In [ ]:
# Plot & table

def _shades(base_color, n=10, min_alpha=0.25, max_alpha=1.0):
    r, g, b = to_rgb(base_color)
    alphas = np.linspace(min_alpha, max_alpha, n)
    return [(r, g, b, float(a)) for a in alphas]


def plot_compliance_trends(reps_ts, title="", figsize=(6, 4), lw=1.2):
    plt.figure(figsize=figsize)
    level_info = [
        ("compliance_level_1", "green"),
        ("compliance_level_2", "yellow"),
        ("compliance_level_3", "orange"),
        ("compliance_level_4", "red"),
    ]
    for col, base in level_info:
        shades = _shades(base, n=len(reps_ts))
        for i, ts in enumerate(reps_ts):
            plt.plot(ts["tick"], ts[col], color=shades[i], linewidth=lw)

    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.xlabel("Tick")
    plt.ylabel("Number of FBOs")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def _mean_ci_table(df, index, columns, mean_col, ci_col):
    tmp = df.copy()
    tmp["value"] = tmp[mean_col].round(1).astype(str) + " ± " + tmp[ci_col].round(1).astype(str)
    tbl = tmp.pivot_table(index=index, columns=columns, values="value", aggfunc="first")
    try:
        tbl = tbl[sorted(tbl.columns, key=lambda x: float(x))]
    except Exception:
        pass
    return tbl


def plot_three_scenarios_compliance(
    reps_by_scenario,
    param_label,
    param_value,
    scenarios=("random", "equal", "tuned"),
    letters=("A", "B", "C"),
    n_fbo=N_FBO,
):
    fig, axes = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(6, 10), constrained_layout=False)

    level_info = [
        ("compliance_level_1", "green",  "Level 1"),
        ("compliance_level_2", "yellow", "Level 2"),
        ("compliance_level_3", "orange", "Level 3"),
        ("compliance_level_4", "red",    "Level 4"),
    ]

    for ax, sc, letter in zip(axes, scenarios, letters):
        reps_ts = reps_by_scenario[sc]

        for col, base_color, _ in level_info:
            shades = _shades(base_color, n=len(reps_ts))
            for i, ts in enumerate(reps_ts):
                ax.plot(ts["tick"], ts[col], color=shades[i], linewidth=1.1)

        ax.set_ylabel("Number of FBOs")
        ax.set_ylim(0, n_fbo * 1.05)  # small headroom
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

        ax.set_title(f"{sc} scenario", loc="left", fontsize=10)
        ax.text(0.01, 0.95, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")

    axes[-1].set_xlabel("Tick")

    fig.suptitle(f"Compliance distributions — {param_label} = {param_value}", fontsize=12)

    legend_lines = [Line2D([0], [0], color=c, lw=2, label=lbl) for _, c, lbl in level_info]
    fig.legend(handles=legend_lines, loc="center left", bbox_to_anchor=(1.03, 0.5),
               borderaxespad=0.0, title="Compliance level")

    fig.tight_layout(rect=(0, 0, 0.80, 0.95))

    safe_val = str(param_value).replace(".", "p")
    filename = os.path.join(FIG_DIR, f"compliance_{param_label}_{safe_val}.png")
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {filename}")

In [ ]:
# Run SA

df_delta, ts_delta = run_sensitivity_delta(
    delta_grid=DELTA_GRID,
    scenarios=SCENARIOS,
    n_reps=N_REPS,
    n_fbo=N_FBO,
)

df_improve, ts_improve = run_sensitivity_improve(
    improve_grid=IMPROVE_GRID,
    scenarios=SCENARIOS,
    n_reps=N_REPS,
    n_fbo=N_FBO,
)

delta_table = _mean_ci_table(
    df_delta, index="scenario", columns="deterioration_step",
    mean_col="caught_rate_mean_%", ci_col="ci_caught_rate_%"
)

improve_table = _mean_ci_table(
    df_improve, index="scenario", columns="improvement_step",
    mean_col="caught_rate_mean_%", ci_col="ci_caught_rate_%"
)


In [ ]:
# Output

# caught-rate tables

display(Markdown("## Caught rate — deterioration_step grid (mean ± 95% CI)"))
display(delta_table)

display(Markdown("## Caught rate — improvement_step grid (mean ± 95% CI)"))
display(improve_table)

print("\n(Plain text versions)")
print("\n--- deterioration_step grid ---")
print(delta_table.to_string())
print("\n--- improvement_step grid ---")
print(improve_table.to_string())

# Combined 3-panel figures 

def _display_saved_png(path):
    try:
        from IPython.display import Image
        display(Image(filename=path))
    except Exception as e:
        print(f"Could not display {path} in notebook: {e}")

display(Markdown("## Combined compliance figures (3 scenarios × 4 levels)"))

# deterioration_step figures
for d in DELTA_GRID:
    reps_by_scenario = {sc: ts_delta[(d, sc)] for sc in SCENARIOS}
    plot_three_scenarios_compliance(
        reps_by_scenario=reps_by_scenario,
        param_label="deterioration_step",
        param_value=d,
        scenarios=SCENARIOS,
        letters=("A", "B", "C"),
        n_fbo=N_FBO,
    )
    safe_val = str(d).replace(".", "p")
    png_path = os.path.join(FIG_DIR, f"compliance_deterioration_step_{safe_val}.png")
    _display_saved_png(png_path)

# improvement_step figures
for step in IMPROVE_GRID:
    reps_by_scenario = {sc: ts_improve[(step, sc)] for sc in SCENARIOS}
    plot_three_scenarios_compliance(
        reps_by_scenario=reps_by_scenario,
        param_label="improvement_step",
        param_value=step,
        scenarios=SCENARIOS,
        letters=("A", "B", "C"),
        n_fbo=N_FBO,
    )
    safe_val = str(step).replace(".", "p")
    png_path = os.path.join(FIG_DIR, f"compliance_improvement_step_{safe_val}.png")
    _display_saved_png(png_path)

# Excel export (summaries + all time series)

excel_out = EXCEL_OUT 

with pd.ExcelWriter(excel_out) as writer:
    # Summary tables
    df_delta.to_excel(writer, sheet_name="delta_summary", index=False)
    df_improve.to_excel(writer, sheet_name="improve_summary", index=False)

    # Delta time-series (one sheet per delta / scenario / repetition)
    for (d, sc), reps in ts_delta.items():
        for i, df_ts in enumerate(reps, start=1):
            sheet_name = f"d_{str(d).replace('.', 'p')}_{sc}_{i}"
            sheet_name = sheet_name[:31]  # Excel limit
            df_ts.to_excel(writer, sheet_name=sheet_name, index=False)

    # Improve time-series (one sheet per step / scenario / repetition)
    for (step, sc), reps in ts_improve.items():
        for i, df_ts in enumerate(reps, start=1):
            sheet_name = f"im_{step}_{sc}_{i}"
            sheet_name = sheet_name[:31]
            df_ts.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Excel file written: {excel_out}")
